![WEkEO Banner](https://github.com/wekeo/frameworks/blob/main/img/WEkEO_banner_partners.png?raw=true)

<div style="display: flex; justify-content: space-between;">
  <a href="./01_Introduction_to_DASK_Gateway.ipynb">&lt;&lt; Introduction to DASK Gateway</a>
  <a href="./DASK_Workers_Processing_Result.ipynb">DASK Gateway Performance Comparison &gt;&gt;</a>
</div>

Copyright (c) 2025 European Union 

License: MIT

# Launch DASK Cluster in the GUI

**This notebook demonstrates how to launch a DASK cluster using the WEkEO GUI and then connect to it from within your Jupyter notebook.**

[![launch-WEKEO](https://img.shields.io/badge/launch-WEKEO-1a4696.svg)](https://jupyterhub.prod.wekeo2.eu/TOBEFILLED)

**PREREQUISITES**

This notebook has the following prerequisites:
- [A WEkEO account](https://my.wekeo.eu/user-registration) 
- Access to the WEkEO JupyterHub workspace
- Basic knowledge of Python and Jupyter notebooks
- Familiarity with the WEkEO workspace GUI

**LEARNING OUTCOMES**

At the end of this notebook you will know: 

- How to launch a DASK cluster using the WEkEO GUI
- How to obtain the cluster URL from the DASK dashboard
- How to connect to a running DASK cluster from your notebook
- How to execute parallel computations using an externally launched cluster

**CONTENTS**

1. [Open the DASK Gateway GUI](#1-step-1-open-the-dask-gateway-gui)  
2. [Launch a New Cluster](#2-launch-a-new-cluster)  
3. [Monitor Cluster Creation](#3-monitor-cluster-creation)  
4. [Obtain the Cluster URL](#4-obtain-the-cluster-url)  
5. [Connect to the Cluster](#5-connect-to-the-cluster)  
6. [Run Parallel Computations](#6-run-parallel-computations)  
7. [Manage and Close Your Cluster](#7-manage-and-close-your-cluster)  
8. [Take home messages](#8-take-home-messages)

<hr>

## 1. Open the DASK Gateway GUI

To launch a DASK cluster using the GUI, follow these steps:

1. In the left sidebar of the WEkEO workspace, look for the **DASK Gateway** icon or link
2. Click on it to open the DASK Gateway GUI

**Screenshot:**

![Open DASK GUI](./img/01-open-dask-gui.png)

## 2. Launch a New Cluster

Once the DASK Gateway GUI is open:

1. Click the **"New Cluster"** button
2. Configure your cluster settings:
   - **Worker Cores**: Select the number of CPU cores per worker
   - **Worker Memory**: Select the amount of RAM per worker (in GB)
3. Click **"Launch"** to start your cluster

**Screenshot:**

<img src="./img/02-launch-new-cluster.png" width="450"/>

## 3. Monitor Cluster Creation

The DASK cluster will now start launching. You'll see a status indicator showing:

- **Cluster Status**: Pending → Starting → Running
- **Worker Count**: Shows how many workers are available
- **Resource Information**: Display of configured cores and memory

Wait until the status shows **"Running"** and your workers are available.

**Screenshot:**

<img src="./img/03-dask-cluster-lauched.png" width="450"/>

## 4. Obtain the Cluster URL

Once your cluster is running:

1. Click on the **"Dashboard"** button or link to open the DASK dashboard

**Screenshot:**

<img src="./img/04-opening-das-dashboard.png" width="450"/>

2. In the DASK dashboard, look for the **Cluster URL** (scheduler address)
3. Copy this URL - you'll need it to connect from your notebook

**Screenshots:**


<img src="./img/06-dask-cluster-url.png" width="450"/>
<img src="./img/05-dask-dashboard.png" width="600"/>



:::{note}
The cluster URL typically looks like: `ws-jupyterhub......`
:::

## 5. Connect to the Cluster

Now that your cluster is running and you have its URL, you can connect to it from within your notebook.

### 5.1 Import required libraries

In [1]:
# Import DASK libraries
from dask.distributed import Client
from dask_gateway import Gateway

# Import DASK and related libraries
import dask.array as da
import xarray as xr
import numpy as np

# For displaying cluster information
from IPython.display import display

### 5.2 Connect to your running cluster

In [3]:
from dask_gateway import Gateway
from dask.distributed import Client
from IPython.display import display

# Connect to the Gateway
gateway = Gateway()

# List clusters
clusters = gateway.list_clusters()

if not clusters:
    raise RuntimeError("No DASK clusters found. Please launch a cluster using the WEkEO GUI first.")

# Most recent cluster
cluster = clusters[0]

print(f"Found {len(clusters)} cluster(s). Connecting to: {cluster.name}")
print(f"Cluster status: {cluster.status}")

# Connect
cluster = gateway.connect(cluster.name)

# Create client
client = Client(cluster)

print("\nSuccessfully connected to the DASK cluster!")
print(f"Dashboard link: {client.dashboard_link}")

display(client)

Found 1 cluster(s). Connecting to: ws-jupyterhub.12a56427fe424dd6836200af4b3c78de
Cluster status: 2

Successfully connected to the DASK cluster!
Dashboard link: /services/dask-gateway/clusters/ws-jupyterhub.12a56427fe424dd6836200af4b3c78de/status


/opt/conda/lib/python3.11/site-packages/distributed/client.py:1393: VersionMismatchWarning: Mismatched versions found

+---------+-----------------+----------------+----------------+
| Package | Client          | Scheduler      | Workers        |
+---------+-----------------+----------------+----------------+
| python  | 3.11.11.final.0 | 3.11.9.final.0 | 3.11.9.final.0 |
| tornado | 6.4.1           | 6.4            | 6.4            |
+---------+-----------------+----------------+----------------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


Connection method: Cluster object,Cluster type: dask_gateway.GatewayCluster
Dashboard: /services/dask-gateway/clusters/ws-jupyterhub.12a56427fe424dd6836200af4b3c78de/status,


### 5.3 Verify worker availability

In [4]:
# Get information about the connected cluster
scheduler_info = client.scheduler_info()

print(f"Cluster status:")
print(f"  Total workers: {len(scheduler_info['workers'])}")
print(f"  Total cores: {sum(w['nthreads'] for w in scheduler_info['workers'].values())}")
print(f"  Total memory: {sum(w['memory_limit'] for w in scheduler_info['workers'].values()) / 1e9:.2f} GB")

print(f"\nWorker details:")
for worker_id, worker_info in scheduler_info['workers'].items():
    print(f"  {worker_id}:")
    print(f"    Cores: {worker_info['nthreads']}")
    print(f"    Memory: {worker_info['memory_limit'] / 1e9:.2f} GB")

Cluster status:
  Total workers: 2
  Total cores: 2
  Total memory: 2.15 GB

Worker details:
  tls://10.42.49.120:34875:
    Cores: 1
    Memory: 1.07 GB
  tls://10.42.60.135:40289:
    Cores: 1
    Memory: 1.07 GB


## 6. Run Parallel Computations

Now that you're connected to your cluster, you can perform parallel computations. This section demonstrates the same computations as in the Introduction to DASK Gateway notebook.

### 6.1 Create a large DASK array

In [6]:
# Define dimensions (time, latitude, longitude)
time_steps = 365    # One year of daily data
lat_points = 500    # 500 latitude points
lon_points = 500    # 500 longitude points

# Create a large random array with DASK
# The 'chunks' parameter defines how the array is divided for parallel processing
large_array = da.random.random(
    (time_steps, lat_points, lon_points), 
    chunks=(50, 100, 100)  # Process in smaller blocks
)

print(f"Array shape: {large_array.shape}")
print(f"Array size in memory: {large_array.nbytes / 1e9:.2f} GB")
print(f"Chunk size: {large_array.chunksize}")
print(f"Number of chunks: {large_array.npartitions}")

Array shape: (365, 500, 500)
Array size in memory: 0.73 GB
Chunk size: (50, 100, 100)
Number of chunks: 200


:::{note}
Notice that creating the array was instant! This is because DASK uses **lazy evaluation** - the array hasn't actually been created yet, just the recipe for creating it.
:::

### 6.2 Perform a computation

In [7]:
%%time

# Calculate the mean over the time dimension
# This operation is still lazy - no computation yet!
mean_over_time = large_array.mean(axis=0)

print("Mean calculation defined (lazy operation)")
print(f"Result shape will be: {mean_over_time.shape}")

Mean calculation defined (lazy operation)
Result shape will be: (500, 500)
CPU times: user 2.42 ms, sys: 92 μs, total: 2.51 ms
Wall time: 2.45 ms


In [8]:
%%time

# Now we actually compute the result
# This triggers the parallel execution across workers
result = mean_over_time.compute()

print(f"\nComputation complete!")
print(f"Result shape: {result.shape}")
print(f"Result sample value: {result[0, 0]:.6f}")


Computation complete!
Result shape: (500, 500)
Result sample value: 0.493506
CPU times: user 32 ms, sys: 11.7 ms, total: 43.7 ms
Wall time: 1.35 s


:::{note}
Watch the DASK dashboard while the computation runs! You'll see:
- Tasks being distributed to workers
- Memory usage increasing as data is processed
- Progress bars showing task completion
:::

### 6.3 Working with xarray and DASK

In [9]:
# Create coordinate arrays
time = np.arange(0, 365)
lat = np.linspace(-90, 90, 500)
lon = np.linspace(-180, 180, 500)

# Create a DASK-backed xarray DataArray
data_dask = da.random.random((365, 500, 500), chunks=(50, 100, 100))

ds = xr.DataArray(
    data_dask,
    dims=("time", "lat", "lon"),
    coords={"time": time, "lat": lat, "lon": lon},
    name="temperature"
)

print("Created xarray DataArray with DASK backend:")
display(ds)

Created xarray DataArray with DASK backend:


<xarray.DataArray 'temperature' (time: 365, lat: 500, lon: 500)> Size: 730MB
dask.array<random_sample, shape=(365, 500, 500), dtype=float64, chunksize=(50, 100, 100), chunktype=numpy.ndarray>
Coordinates:
  * time     (time) int64 3kB 0 1 2 3 4 5 6 7 ... 358 359 360 361 362 363 364
  * lat      (lat) float64 4kB -90.0 -89.64 -89.28 -88.92 ... 89.28 89.64 90.0
  * lon      (lon) float64 4kB -180.0 -179.3 -178.6 -177.8 ... 178.6 179.3 180.0

In [10]:
%%time

# Perform a realistic analysis: select a region and compute statistics
# Select a region (e.g., Europe)
europe = ds.sel(lat=slice(35, 70), lon=slice(-10, 40))

# Calculate temporal mean and standard deviation
temporal_mean = europe.mean(dim="time").compute()
temporal_std = europe.std(dim="time").compute()

print("Regional statistics computed!")
print(f"Mean value range: {float(temporal_mean.min()):.4f} to {float(temporal_mean.max()):.4f}")
print(f"Std dev range: {float(temporal_std.min()):.4f} to {float(temporal_std.max()):.4f}")

Regional statistics computed!
Mean value range: 0.4242 to 0.5628
Std dev range: 0.2634 to 0.3130
CPU times: user 38.9 ms, sys: 6.18 ms, total: 45.1 ms
Wall time: 541 ms


## 7. Manage and Close Your Cluster

### 7.1 Monitoring cluster resources

You can check the status of your cluster and workers anytime:

In [11]:
# Get current scheduler information
scheduler_info = client.scheduler_info()

print(f"Current cluster status:")
print(f"  Total workers: {len(scheduler_info['workers'])}")
print(f"  Total cores: {sum(w['nthreads'] for w in scheduler_info['workers'].values())}")
print(f"  Total memory: {sum(w['memory_limit'] for w in scheduler_info['workers'].values()) / 1e9:.2f} GB")

Current cluster status:
  Total workers: 2
  Total cores: 2
  Total memory: 2.15 GB


### 7.2 Close your client connection

When you're done with your computations, close the client connection. The cluster itself remains running in the WEkEO GUI until you explicitly terminate it there.

In [12]:
# Close the client connection
client.close()
print("Client disconnected from the cluster")

Client disconnected from the cluster


:::{warning}
The cluster continues running even after you close the client. To fully terminate the cluster and free up resources:

1. Return to the WEkEO DASK Gateway GUI
2. Find your cluster in the list
3. Click the **"Shutdown"** or **"Delete"** button to terminate it

Always remember to shut down your cluster when finished to free up resources for other users!
:::

## 8. Take home messages

**Key takeaways from this notebook:**

1. **GUI cluster launch is straightforward** - Use the WEkEO DASK Gateway GUI for easy cluster creation

2. **Get the cluster URL from the dashboard** - This is your connection point for notebook-based operations

3. **Connect externally launched clusters** - Use `Client(url)` to connect to clusters started via the GUI

4. **Same code, external management** - Once connected, use the same DASK code as before

5. **Monitor via the dashboard** - The DASK dashboard provides real-time insights into your computations

6. **Always clean up** - Close clients and shut down clusters in the GUI when finished

7. **Flexible workflow** - Start clusters via GUI for stability, or create them programmatically for automation

**Advantages of GUI-based cluster launching:**

- Visual feedback and real-time status monitoring
- Easy configuration adjustments without code
- Cluster persistence across multiple notebook sessions
- Better resource management visibility

## 9. Next steps

Now that you know both programmatic and GUI-based cluster launching:

- **Explore performance comparisons**: Check out [03_Performance_benchmark_of_DASK_Gateway.ipynb](./03_Performance_benchmark_of_DASK_Gateway.ipynb) to see detailed performance benchmarks
- **Review the programmatic approach**: Return to [01_Introduction_to_DASK_Gateway.ipynb](./01_Introduction_to_DASK_Gateway.ipynb) for the programmatic cluster creation method
- **Apply DASK to real data**: Try using DASK with actual Copernicus data from the WEkEO Data Store
- **Optimize your workflows**: Experiment with different chunk sizes and worker configurations for your specific use cases

### Additional resources:

- [DASK Documentation](https://docs.dask.org/)
- [DASK Best Practices](https://docs.dask.org/en/latest/best-practices.html)
- [xarray with DASK](https://docs.xarray.dev/en/stable/user-guide/dask.html)
- [WEkEO Help Center](https://help.wekeo.eu/)

<hr>

[View on GitHub](https://github.com/wekeo) | 
[Visit WEkEO](https://wekeo.copernicus.eu) | 
[Run Notebooks on the WEkEO Workspace](https://jupyterhub.prod.wekeo2.eu/) | 
[Contact helpdesk for support](mailto:support@wekeo.eu)